In [11]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.metrics import confusion_matrix, cohen_kappa_score, accuracy_score
from PIL import Image, ImageFilter
import timm

# --- SETTINGS ---
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
BASE_PATH = '/kaggle/input/datasets/mariaherrerot/aptos2019/'
TRAIN_IMG_DIR = os.path.join(BASE_PATH, 'train_images/train_images')
VAL_IMG_DIR = os.path.join(BASE_PATH, 'val_images/val_images')
IMG_SIZE = 224
BATCH_SIZE = 16 # Reduced for Mixup overhead
ALPHA = 0.2     # Mixup strength

# --- NOVELTY: MIXUP AUGMENTATION ---
def mixup_data(x, y, alpha=1.0, device='cuda'):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# --- ARCHITECTURE (Optimized Ensemble) ---
class MasterAptosEnsemble(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = timm.create_model('convnext_tiny', pretrained=True, num_classes=5)
        self.vit = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=5)
        
    def forward(self, x):
        return (self.cnn(x) + self.vit(x)) / 2

# --- DATASET WITH SHARPNESS NOVELTY ---
class AptosMasterDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(os.path.join(BASE_PATH, csv_file))
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, f"{self.df.iloc[idx, 0]}.png")
        image = Image.open(img_path).convert('RGB')
        # Apply slight sharpness to highlight lesions
        image = image.filter(ImageFilter.SHARPEN)
        label = self.df.iloc[idx, 1]
        if self.transform: image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

def run_master_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MasterAptosEnsemble().to(device)
    
    # SWA setup
    swa_model = torch.optim.swa_utils.AveragedModel(model)
    optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.05)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    train_loader = DataLoader(AptosMasterDataset('train_1.csv', TRAIN_IMG_DIR, train_tf), 
                              batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(AptosMasterDataset('valid.csv', VAL_IMG_DIR, train_tf), batch_size=BATCH_SIZE)

    print("Initiating Final SWA + Mixup Training...")
    
    for epoch in range(15):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            
            # Apply Mixup
            imgs, labels_a, labels_b, lam = mixup_data(imgs, labels, ALPHA, device)
            
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            loss.backward()
            optimizer.step()
        
        # Start SWA averaging after epoch 10
        if epoch > 10:
            swa_model.update_parameters(model)

        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                out = model(imgs)
                preds.extend(torch.argmax(out, 1).cpu().numpy())
                targets.extend(labels.cpu().numpy())
        
        print(f"Epoch {epoch+1} | Val Acc: {accuracy_score(targets, preds):.4%}")

    # Final Evaluation with SWA model
    print("\n" + "="*50)
    print("      FINAL SWA-ENSEMBLE CLINICAL REPORT")
    print("="*50)
    swa_model.eval()
    f_preds, f_targets = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = swa_model(imgs)
            f_preds.extend(torch.argmax(out, 1).cpu().numpy())
            f_targets.extend(labels.cpu().numpy())

    final_acc = accuracy_score(f_targets, f_preds)
    print(f"SWA Accuracy: {final_acc:.4%}")
    print(f"Final Kappa:  {cohen_kappa_score(f_targets, f_preds, weights='quadratic'):.4f}")
    
    cm = confusion_matrix(f_targets, f_preds)
    classes = ['No DR', 'Mild', 'Moderate', 'Severe', 'Prolif']
    for i in range(5):
        tp = cm[i, i]
        fn = sum(cm[i, :]) - tp
        fp = sum(cm[:, i]) - tp
        tn = sum(cm.flatten()) - (tp + fn + fp)
        print(f"{classes[i]:<10} | Sens: {tp/(tp+fn):.4f} | Spec: {tn/(tn+fp):.4f}")
    
    torch.save(swa_model.state_dict(), 'swa_ensemble_final.pth')

if __name__ == "__main__":
    run_master_training()

Initiating Final SWA + Mixup Training...
Epoch 1 | Val Acc: 74.5902%
Epoch 2 | Val Acc: 79.7814%
Epoch 3 | Val Acc: 80.3279%
Epoch 4 | Val Acc: 82.2404%
Epoch 5 | Val Acc: 83.0601%
Epoch 6 | Val Acc: 83.8798%
Epoch 7 | Val Acc: 81.4208%
Epoch 8 | Val Acc: 84.1530%
Epoch 9 | Val Acc: 80.8743%
Epoch 10 | Val Acc: 82.2404%
Epoch 11 | Val Acc: 82.7869%
Epoch 12 | Val Acc: 76.5027%
Epoch 13 | Val Acc: 83.3333%
Epoch 14 | Val Acc: 86.3388%
Epoch 15 | Val Acc: 83.8798%

      FINAL SWA-ENSEMBLE CLINICAL REPORT
SWA Accuracy: 84.9727%
Final Kappa:  0.9130
No DR      | Sens: 0.9942 | Spec: 0.9845
Mild       | Sens: 0.6500 | Spec: 0.9755
Moderate   | Sens: 0.8365 | Spec: 0.8969
Severe     | Sens: 0.3636 | Spec: 0.9797
Prolif     | Sens: 0.6786 | Spec: 0.9704
